In [1]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class CNN(nn.Module):
    """Convolutional feature extractor with progressive kernel reduction."""
    def __init__(self, in_channels=1, input_size=(80, 80), n_conv=12, conv_channels=64, pool_every=3):
        super().__init__()
        assert 10 <= n_conv <= 15 #as in the nature paper
        
        max_k = 7
        ks = []
        for i in range(n_conv):
            k = int(round(max_k - (max_k - 3) * (i / (n_conv - 1))))
            if k % 2 == 0:
                k += 1
            ks.append(k)

        layers = []
        cur_ch = in_channels
        for i, k in enumerate(ks):
            pad = k // 2
            layers += [
                nn.Conv2d(cur_ch, conv_channels, kernel_size=k, padding=pad),
                nn.BatchNorm2d(conv_channels),
                nn.LeakyReLU(0.01, inplace=True)
            ]
            cur_ch = conv_channels
            if (i + 1) % pool_every == 0 and (i + 1) != n_conv:
                layers.append(nn.MaxPool2d(2, 2, ceil_mode=True))

        self.conv_stack = nn.Sequential(*layers)

        # determine flatten size
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, *input_size)
            out = self.conv_stack(dummy)
            self.flatten_dim = out.numel()


    def forward(self, x):
        feat = self.conv_stack(x)
        return feat.view(x.size(0), -1)


class FCNet(nn.Module):
    """Fully connected layers with batch-norm, LeakyReLU, and dropout."""
    def __init__(self, input_dim, output_dim, n_fc=6, dropout_p=0.5): #the paper suggest n_fc=5/6
        super().__init__()
        hidden_dims = [max(256, input_dim // (2 ** (i + 1))) for i in range(n_fc - 1)]
        dims = [input_dim] + hidden_dims + [output_dim]

        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers += [
                    nn.BatchNorm1d(dims[i + 1]),
                    nn.LeakyReLU(0.01, inplace=True),
                    nn.Dropout(p=dropout_p)
                ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class CombinedModel(nn.Module):
    """Combined CNN + FC network."""
    def __init__(self, img_channels=1, img_size=(80, 80), n_conv=12, conv_channels=64,
                 pool_every=3, n_fc=6, fc_output_dim=256, dropout_p=0.5):
        super().__init__()
        self.cnn = CNN(in_channels=img_channels, input_size=img_size,
                       n_conv=n_conv, conv_channels=conv_channels, pool_every=pool_every)
        self.fc = FCNet(self.cnn.flatten_dim, fc_output_dim, n_fc=n_fc, dropout_p=dropout_p)

    def forward(self, x):
        x = self.cnn(x)
        return self.fc(x)


if __name__ == "__main__":
    model = CombinedModel(fc_output_dim=256).to(device)
    v = torch.randn(2, 1, 80, 80).to(device)
    y = model(v)
    print("Input:", v.shape)
    print("Output:", y.shape)



Input: torch.Size([2, 1, 80, 80])
Output: torch.Size([2, 256])


In [ ]:
#overfitting a single batch test
import torch
import torch.nn as nn
from dataloader import build_loaders


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


data_root = "."   
train_loader, test_loader, dataset = build_loaders(data_root, n=2, batch_size=2)


xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
print(f"Input batch shape: {xb.shape}")
print(f"Output batch shape: {yb.shape}")


model = CombinedModel(
    img_channels=1,
    img_size=(80, 80),  
    n_conv=12,
    conv_channels=64,
    pool_every=3,
    n_fc=6,
    fc_output_dim=yb.shape[1],
    dropout_p=0.3
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for step in range(2000):
    pred = model(xb)
    loss = criterion(pred, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"Step {step:03d} | Loss = {loss.item():.6f}")

print("Overfit test finished.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from dataloader import build_loaders
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === 1. Load data ===
train_loader, val_loader, dataset = build_loaders(
    data_root = "."  ,
    n=1,#modify this one for higher dim output
    batch_size=4,
    test_ratio=0.1,  # 9:1 split
    seed=42
)

# === 2. Build model ===
model = CombinedModel(
    img_channels=1,
    img_size=(80, 80),
    n_conv=12,
    conv_channels=64,
    pool_every=3,
    n_fc=6,
    fc_output_dim=2 * 32, #modify this one (2**n * 32) for higher dim output
    dropout_p=0.5
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)

best_val_loss = float('inf')
patience = 100
no_improve_epochs = 0
num_epochs = 1000

train_losses, val_losses = [], []

# === 2. Training loop ===
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)

    # === Validation ===
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)
    val_losses.append(val_loss)

    scheduler.step()

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # === Early stopping ===
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve_epochs = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            print("Early stopping triggered.")
            break

print("Training done. Best validation loss:", best_val_loss)

# === 3. Plot losses ===
plt.figure(figsize=(6,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training vs Validation Loss")
plt.tight_layout()
plt.show()

In [ ]:
'''
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
'''